## Direct API to Workflows

This final lesson pulls all your skills together, showing you how to turn a series of disconnected steps into a professional, production-ready system. Here is the content formatted for clarity and impact.

---

# Introduction: Direct API Gateway to Workflows

Welcome to the final lesson of this course on building serverless applications! So far, you have learned how to use Google Cloud Functions, API Gateway, and Workflows to create and deploy serverless logic. In this lesson, we take your skills a step further by showing you how to use API Gateway to start Workflows directly—without needing an extra "middleman" Cloud Function just to trigger a workflow.



This approach makes your applications faster and more cost-effective. You will learn how to use Python code to:
* **Control** the request and response.
* **Validate** input before the workflow starts.
* **Handle errors** gracefully.
* **Route** to different workflows dynamically.

---

## 1. Generating Unique Execution Names

When starting a Workflow, you want each run to have a unique name for tracking and debugging. In Python, we use the `uuid` module to generate these identifiers.

```python
import uuid

def generate_execution_name(prefix="order"):
    # uuid.uuid4().hex[:8] generates a random 8-character string
    unique_id = uuid.uuid4().hex[:8]
    return f"{prefix}-{unique_id}"

# Example usage:
execution_name = generate_execution_name()
print(execution_name)  # e.g., "order-7f3b2c1a"
```

---

## 2. Input Validation

It is important to ensure the incoming request has all required fields before burning resources on a workflow. We can perform this validation in Python before the execution call.

```python
def validate_input(request):
    request_json = request.get_json()
    
    if not request_json or not request_json.get("customerId"):
        return {"error": "Missing required field: customerId"}, 400
        
    return request_json, 200

# Usage in a trigger function:
def start_workflow(request):
    data, status = validate_input(request)
    if status != 200:
        return data, status
    # Proceed to start the workflow...
```

---

## 3. Transforming Responses

Raw API responses from cloud services aren't always user-friendly. We can use Python to "clean up" the data before it reaches the client.

```python
import datetime

def transform_response(execution_name, execution_id):
    response = {
        "execution_id": execution_id,
        "execution_name": execution_name,
        "started_at": datetime.datetime.utcnow().isoformat() + "Z",
        "status": "RUNNING"
    }
    return response, 202
```

---

## 4. Error Handling

Starting a workflow can fail for many reasons (e.g., duplicate names or internal service errors). We use `try/except` blocks to catch these and return meaningful HTTP codes.

```python
def start_workflow(request):
    try:
        # ... validation and start logic ...
        execution_name = generate_execution_name()
        execution_id = f"projects/demo/locations/us/workflows/OrderProcessing/executions/{execution_name}"
        
        return transform_response(execution_name, execution_id)
        
    except ValueError as e:
        return {"error": "Invalid request parameters"}, 400
    except Exception as e:
        return {
            "error": "Internal server error",
            "message": str(e)
        }, 500
```

---

## 5. Dynamic Workflow Routing

You can use a single API endpoint to trigger different workflows (e.g., orders vs. refunds) based on the request body.

```python
def start_workflow(request):
    request_json = request.get_json()
    
    # Check for customerId
    if not request_json or not request_json.get("customerId"):
        return {"error": "Missing required field: customerId"}, 400
    
    # Determine which workflow type to run
    workflow_type = request_json.get("workflow_type", "order")
    if workflow_type not in ["order", "refund", "inventory"]:
        workflow_type = "order"

    execution_name = generate_execution_name(prefix=workflow_type)
    
    # Logic to select the target workflow based on workflow_type would go here
    execution_id = f"projects/demo/locations/us/workflows/OrderProcessing/executions/{execution_name}"
    
    return transform_response(execution_name, execution_id)
```

---

## Summary and Next Steps

Congratulations! You have reached the end of this course on building serverless applications with Google Cloud. You have mastered:
1. **Naming:** Generating unique identifiers with `uuid`.
2. **Validation:** Protecting your workflows from bad data.
3. **Transformation:** Making API responses developer-friendly.
4. **Routing:** Using one endpoint to handle multiple business processes.

You are now ready for the final hands-on exercises. Great work, and keep building!

## Fixing Static Execution Names for Workflow Runs in Python

You are now ready to practice generating unique execution names for Workflow executions in Python.

Currently, the provided Python function for starting a workflow uses a static execution name for every request. This can cause failures if multiple requests are made at the same time, since each execution name must be unique.

Your task is to fix this by modifying the code to generate a unique execution name for each request. Specifically:

    Find the # TODO comment in the start_workflow function in main.py.
    Replace the static execution name with a dynamic one using the generate_execution_name function.
    Use the "order-" prefix for the execution name.

This change will make your API more reliable and allow multiple concurrent workflow executions. Once you complete this, your direct integration between API Gateway and Workflows will be robust!

```python
import uuid
import datetime

def generate_execution_name(prefix="order"):
    unique_id = uuid.uuid4().hex[:8]
    return f"{prefix}-{unique_id}"

def transform_response(execution_name, execution_id):
    response = {
        "execution_id": execution_id,
        "execution_name": execution_name,
        "started_at": datetime.datetime.utcnow().isoformat() + "Z",
        "status": "RUNNING"
    }
    return response

def start_workflow(request):
    request_json = request.get_json(silent=True)
    if not request_json or not request_json.get("customerId"):
        return {"error": "Missing required field: customerId"}, 400

    # TODO: Replace the static execution name with a unique one using generate_execution_name and "order-" prefix
    execution_name = "static-order-name"

    execution_id = f"projects/demo/locations/us-central1/workflows/OrderProcessing/executions/{execution_name}"

    return transform_response(execution_name, execution_id), 202

```

Using a static execution name is a common pitfall—it works for the first test, but the moment you have two concurrent users, the system throws a "Conflict" error. By implementing dynamic naming, you're making your API truly scalable.

Here is the fixed code with the unique execution name implementation:

```python
import uuid
import datetime

def generate_execution_name(prefix="order"):
    unique_id = uuid.uuid4().hex[:8]
    return f"{prefix}-{unique_id}"

def transform_response(execution_name, execution_id):
    response = {
        "execution_id": execution_id,
        "execution_name": execution_name,
        "started_at": datetime.datetime.utcnow().isoformat() + "Z",
        "status": "RUNNING"
    }
    return response

def start_workflow(request):
    request_json = request.get_json(silent=True)
    if not request_json or not request_json.get("customerId"):
        return {"error": "Missing required field: customerId"}, 400

    # FIX: Replaced the static string with a call to the generator function
    execution_name = generate_execution_name(prefix="order")

    execution_id = f"projects/demo/locations/us-central1/workflows/OrderProcessing/executions/{execution_name}"

    return transform_response(execution_name, execution_id), 202
```

### Why this is the "Robust" way:
1.  **Concurrency:** Two orders placed at the exact same millisecond will now have different suffixes (e.g., `order-a1b2c3d4` and `order-e5f6g7h8`), allowing both to process.
2.  **Traceability:** In the Google Cloud Console, you can now search for specific execution names that match your internal order IDs or logs.
3.  **Efficiency:** You are generating this name *after* the validation check. This ensures you don't waste CPU cycles or memory generating UUIDs for requests that are just going to be rejected anyway.

## Add Input Validation for customerId in Python API

Your order processing workflow currently validates that the order amount is greater than zero, but it doesn't check if a customerId is provided. This can lead to incomplete orders being processed.

Your task is to add input validation to the validate_order function in main.py so that it checks for the presence of the customerId field in the order. If customerId is missing or empty, the function should return a 400 error with a helpful message. If the input is valid, the validation should proceed as normal.

Specifically:

    Find the #TODO comment in the main.py file where the validation logic should be added.
    Add code to check if customerId is missing or empty in the order data.
    If validation fails, return a 400 error and a JSON message: {"error": "Missing required field: customerId"}
    If validation passes, continue to validate the order amount as before.

This will make your workflow more robust and prevent incomplete orders from being processed.

Test your implementation with the provided event in event.json. The workflow should fail at the validation step when customerId is missing.

```python
def validate_order(request):
    request_json = request.get_json()
    order = request_json["order"]
    # TODO: Add input validation for customerId here
    if float(order["amount"]) <= 0:
        return {"error": "Invalid amount"}, 400
    order["validated"] = True
    return {"order": order}

def charge_customer(request):
    request_json = request.get_json()
    order = request_json["order"]
    request_json["payment_id"] = f"pay_{order['order_id']}"
    request_json["charged"] = True
    return request_json
    
import time
from google.cloud import firestore

def record_order(request):
    db = firestore.Client()
    request_json = request.get_json()
    order = request_json["order"]
    item = {
        "order_id": order["order_id"],
        "amount": float(order["amount"]),
        "payment_id": request_json["payment_id"],
        "ts": int(time.time())
    }
    db.collection("orders").document(order["order_id"]).set(item)
    return {"status": "ok", "order": item}

def notify_customer(request):
    request_json = request.get_json()
    order = request_json["order"]
    message = f"Order confirmation: Order {order['order_id']} for ${order['amount']} has been processed. Payment ID: {order['payment_id']}"
    print(message)
    return {"status": "notification_sent", "message": message}

```

Adding `customerId` validation is a crucial step in ensuring your data integrity. In a production environment, missing identifiers can break database relationships and make tracking orders nearly impossible.

Here is the updated `validate_order` function with the required validation logic.

### Updated `main.py`

```python
def validate_order(request):
    request_json = request.get_json(silent=True) or {}
    order = request_json.get("order", {})
    
    # FIX: Check if customerId exists and is not an empty string or None
    if not order.get("customerId"):
        return {"error": "Missing required field: customerId"}, 400
        
    # Existing amount validation
    if float(order.get("amount", 0)) <= 0:
        return {"error": "Invalid amount"}, 400
        
    order["validated"] = True
    return {"order": order}

# ... (rest of your functions remain the same)
```

---

### Key Improvements Explained

* **Truthiness Check:** Using `if not order.get("customerId"):` is a powerful Pythonic pattern. It catches cases where the key is missing entirely, if the value is `None`, or if it's an empty string `""`.
* **Order of Operations:** We check for the `customerId` **before** the amount. This is intentional—there is no point checking the financial math if we don't even know who the customer is.
* **Fail-Fast Logic:** By returning immediately with a `400` status code, we prevent the workflow from proceeding to the expensive `charge_customer` and `record_order` steps.

### Test with `event.json`

To trigger the validation error and see your fix in action, use a test payload like this:

```json
{
  "order": {
    "order_id": "test-123",
    "amount": "99.99"
  }
}
```

Since `"customerId"` is missing from this JSON, your function will correctly return the error message, and your Workflow `try/except` block will catch it and stop the execution.

## Transforming Workflow API Responses

Now that your workflow processes orders through multiple steps, let's make the final response user-friendly and professional!

Currently, when your workflow completes, it returns the raw response from the last step, which isn't formatted consistently for API consumers. You need to transform this response into a clean, informative JSON structure that includes:

    The execution ID (use the order_id as the execution identifier)
    An execution name in the format "order-execution-{order_id}"
    A timestamp for when the execution completed
    A clear status indicator

Your task: Implement the transform_response function in main.py and add a final transformation step to order_workflow.yaml. Look for the # TODO comments and replace them with code that builds and returns the correct response.

This will make your workflow's API responses much easier to use and understand!

```python
def validate_order(request):
    request_json = request.get_json()
    order = request_json["order"]
    if float(order["amount"]) <= 0:
        return {"error": "Invalid amount"}, 400
    order["validated"] = True
    return {"order": order}

def charge_customer(request):
    request_json = request.get_json()
    order = request_json["order"]
    request_json["payment_id"] = f"pay_{order['order_id']}"
    request_json["charged"] = True
    return request_json
    
import time
from google.cloud import firestore

def record_order(request):
    db = firestore.Client()
    request_json = request.get_json()
    order = request_json["order"]
    item = {
        "order_id": order["order_id"],
        "amount": float(order["amount"]),
        "payment_id": request_json["payment_id"],
        "ts": int(time.time())
    }
    db.collection("orders").document(order["order_id"]).set(item)
    return {"status": "ok", "order": item}

def notify_customer(request):
    request_json = request.get_json()
    order = request_json["order"]
    message = f"Order confirmation: Order {order['order_id']} for ${order['amount']} has been processed. Payment ID: {order['payment_id']}"
    print(message)
    return {"status": "notification_sent", "message": message, "order": order}

import datetime

def transform_response(request):
    # TODO: Extract the request JSON and get the order data
    
    # TODO: Build and return a dictionary response with:
    # - execution_id: the order_id
    # - execution_name: formatted as "order-execution-{order_id}"
    # - started_at: current UTC time in ISO 8601 format, ending with 'Z'
    # - status: set to "COMPLETED"
    pass
```

```yaml
main:
  params: [input]
  steps:
    - validate:
        try:
          call: http.post
          args:
            url: http://localhost:3001
            body: ${input}
          result: validated
        except:
          as: e
          steps:
            - return_error:
                return: ${e}
    - charge:
        try:
          call: http.post
          args:
            url: http://localhost:3002
            body: ${validated.body}
          result: charged
        except:
          as: e
          steps:
            - return_error:
                return: ${e}
    - record:
        try:
          call: http.post
          args:
            url: http://localhost:3003
            body: ${charged.body}
          result: recorded
        except:
          as: e
          steps:
            - return_error:
                return: ${e}
    - notify:
        try:
          call: http.post
          args:
            url: http://localhost:3004
            body: ${recorded.body}
          result: notified
        except:
          as: e
          steps:
            - return_error:
                return: ${e}
    # TODO: Add a transform step here that calls http://localhost:3005
    #       Pass ${notified.body} as the body and store the result as transformed
    #       Include error handling with try/except
    - return_output:
        return: ${notified.body}  # TODO: Change this to return ${transformed.body}
```

```json
{"order": {"order_id": "test-123", "amount": "99.99"}}
```

## Enhanced Error Handling for Workflow Execution API

You've built a robust API for starting Google Workflows directly, with input validation and user-friendly responses. However, to make your API production-ready, you need to handle specific error cases that can occur when starting a workflow execution.

Your task is to enhance the error handling in your Python Cloud Function by adding custom responses for common Google Workflows API errors. Specifically:

    Add a response for ExecutionAlreadyExists errors that returns a 409 Conflict status and a clear, user-friendly message.
    Add a response for WorkflowNotFound errors that returns a 404 Not Found status and a helpful message.
    Add a response for InvalidArgument errors that returns a 400 Bad Request status and a descriptive message.
    Improve the generic error responses to include more helpful information.

Look for the # TODO comments in the main.py file and implement the missing error handling logic. Each error type should map to the correct HTTP status code and return a clear JSON error message to the client.
Suggestions

```python
import uuid
import datetime

def start_workflow(request):
    try:
        request_json = request.get_json()
        data = request_json
        
        if not data or not data.get("customerId"):
            return {"error": "Missing required field: customerId"}, 400

        workflow_type = data.get("workflow_type", "order")
        if workflow_type not in ["order", "refund", "inventory"]:
            workflow_type = "order"

        execution_name = f"{workflow_type}-{uuid.uuid4().hex[:8]}"
        workflow_input = {
            "order": data,
            "workflow_type": workflow_type
        }

        # Simulate starting the workflow and possible errors
        simulate_error = data.get("simulate_error")
        if simulate_error == "ExecutionAlreadyExists":
            # TODO: Return 409 Conflict with a user-friendly error message
            pass
        elif simulate_error == "WorkflowNotFound":
            # TODO: Return 404 Not Found with a clear error message
            pass
        elif simulate_error == "InvalidArgument":
            # TODO: Return 400 Bad Request with a descriptive error message
            pass

        # Simulate successful execution
        execution_id = f"projects/demo/locations/us-central1/workflows/OrderProcessing/executions/{execution_name}"
        response = {
            "execution_id": execution_id,
            "execution_name": execution_name,
            "started_at": datetime.datetime.utcnow().isoformat() + "Z",
            "status": "RUNNING"
        }
        return response

    except ValueError as e:
        # TODO: Return 400 Bad Request with helpful error info
        pass
    except Exception as e:
        # TODO: Return 500 Internal Server Error with error details
        pass
```

Production-grade error handling is about more than just catching a crash; it's about providing the "caller" with actionable information. By mapping specific Google Cloud Workflows API exceptions to standard HTTP status codes, you allow client applications to respond intelligently (e.g., retrying on a 500 but stopping on a 400).

Here is the enhanced `main.py` with the completed error handling logic.

---

## Updated `main.py` with Custom Error Responses

```python
import uuid
import datetime

def start_workflow(request):
    try:
        request_json = request.get_json(silent=True) or {}
        data = request_json
        
        # Initial Validation
        if not data or not data.get("customerId"):
            return {"error": "Missing required field: customerId"}, 400

        workflow_type = data.get("workflow_type", "order")
        if workflow_type not in ["order", "refund", "inventory"]:
            workflow_type = "order"

        execution_name = f"{workflow_type}-{uuid.uuid4().hex[:8]}"

        # Simulate starting the workflow and possible errors based on input
        simulate_error = data.get("simulate_error")
        
        if simulate_error == "ExecutionAlreadyExists":
            # 409 Conflict: The name was already taken
            return {
                "error": "Conflict",
                "message": f"An execution named '{execution_name}' already exists. Please try again."
            }, 409
            
        elif simulate_error == "WorkflowNotFound":
            # 404 Not Found: The workflow definition is missing
            return {
                "error": "Not Found",
                "message": f"The workflow '{workflow_type}' could not be located in this region."
            }, 404
            
        elif simulate_error == "InvalidArgument":
            # 400 Bad Request: The parameters sent were malformed
            return {
                "error": "Bad Request",
                "message": "The workflow input parameters are invalid or malformed."
            }, 400

        # Simulate successful execution
        execution_id = f"projects/demo/locations/us-central1/workflows/OrderProcessing/executions/{execution_name}"
        response = {
            "execution_id": execution_id,
            "execution_name": execution_name,
            "started_at": datetime.datetime.utcnow().isoformat() + "Z",
            "status": "RUNNING"
        }
        return response, 202 # 202 Accepted is standard for starting async tasks

    except ValueError as e:
        # Catch value-specific issues (like float conversion failures)
        return {
            "error": "Invalid Data Format",
            "details": str(e)
        }, 400
        
    except Exception as e:
        # Catch-all for unexpected server-side issues
        return {
            "error": "Internal Server Error",
            "message": "An unexpected error occurred while starting the workflow.",
            "details": str(e)
        }, 500
```

---

## Error Mapping Summary

When building these integrations, it's helpful to visualize how the Cloud Function acts as a translator between the backend Cloud services and the frontend client.



| Error Scenario | HTTP Code | Meaning |
| :--- | :--- | :--- |
| **Missing customerId** | `400` | User error; request was incomplete. |
| **InvalidArgument** | `400` | User error; request was formatted incorrectly. |
| **WorkflowNotFound** | `404` | Infrastructure error; the target process doesn't exist. |
| **ExecutionAlreadyExists**| `409` | Conflict; the unique identifier is already in use. |
| **General Exception** | `500` | Server error; something went wrong in our code. |

### Why this matters
* **UX:** Instead of seeing a generic "Error 500," the user might see "Duplicate Order" (409) or "Invalid Customer ID" (400).
* **Security:** Detailed error messages in the `Exception` block should be logged internally, but the `message` returned to the user should be sanitized so you don't leak sensitive infrastructure details.
* **Automation:** Standard codes like `409` tell automated systems (like a frontend retry-logic) that retrying won't help unless the data is changed.

## Dynamic Workflow Routing in a Python API Handler

You are now ready to make your API flexible by supporting multiple workflow types! Currently, your Python API only handles a single workflow, but real-world systems often need to process orders, refunds, and inventory updates through the same endpoint.

Your task is to implement dynamic workflow routing in your Python API handler. Specifically, you need to:

    Set a default workflow type of "order" for backward compatibility
    Extract and validate the workflow_type parameter from the incoming request body, allowing only "order", "refund", or "inventory"
    Use the workflow_type to generate a dynamic execution name
    Structure the workflow input to include both the original request data (as "order") and the workflow_type

This will turn your single-purpose API into a flexible routing system that can handle multiple workflow types through one endpoint. Use conditional logic and input validation to ensure robust behavior.

```python
import uuid
import datetime

def generate_execution_name(prefix="order"):
    unique_id = uuid.uuid4().hex[:8]
    return f"{prefix}-{unique_id}"

def transform_response(execution_name, execution_id):
    response = {
        "execution_id": execution_id,
        "execution_name": execution_name,
        "started_at": datetime.datetime.utcnow().isoformat() + "Z",
        "status": "RUNNING"
    }
    return response, 202

def start_workflow(request):
    data = request.get_json(silent=True)
    if not data or not data.get("customerId"):
        return {"error": "Missing required field: customerId"}, 400

    # TODO: Set a default workflow_type variable to "order"
    # TODO: Extract workflow_type from request body if it exists and is not empty
    # TODO: Validate that workflow_type is one of: "order", "refund", or "inventory"
    # TODO: Use the workflow_type variable in the execution name instead of hardcoded "order"
    # TODO: Structure the workflow input to include both the original request as "order" and the workflow_type

    execution_name = "order-xxxxxxx"
    # Simulate starting the workflow and getting an execution_id
    execution_id = f"projects/demo/locations/us-central1/workflows/OrderProcessing/executions/{execution_name}"

    return transform_response(execution_name, execution_id)

```

Dynamic workflow routing is a powerful pattern that allows your API to act as a "traffic controller." Instead of managing multiple endpoints for different business processes, you can centralize your logic and dispatch requests to the appropriate workflow based on the payload.

Here is the completed implementation for your flexible routing system.

---

## Updated `main.py` with Dynamic Routing

```python
import uuid
import datetime

def generate_execution_name(prefix="order"):
    unique_id = uuid.uuid4().hex[:8]
    return f"{prefix}-{unique_id}"

def transform_response(execution_name, execution_id):
    response = {
        "execution_id": execution_id,
        "execution_name": execution_name,
        "started_at": datetime.datetime.utcnow().isoformat() + "Z",
        "status": "RUNNING"
    }
    return response, 202

def start_workflow(request):
    data = request.get_json(silent=True) or {}
    
    # Core validation
    if not data or not data.get("customerId"):
        return {"error": "Missing required field: customerId"}, 400

    # 1. Set default and 2. Extract workflow_type from request
    workflow_type = data.get("workflow_type", "order")

    # 3. Validate that workflow_type is an allowed value
    allowed_types = ["order", "refund", "inventory"]
    if workflow_type not in allowed_types:
        # Fallback to default or return error; here we fallback for compatibility
        workflow_type = "order"

    # 4. Use the dynamic prefix for the execution name
    execution_name = generate_execution_name(prefix=workflow_type)
    
    # 5. Structure the workflow input 
    # We include the full data under 'order' and the type separately
    workflow_input = {
        "order": data,
        "workflow_type": workflow_type
    }

    # Simulate starting the workflow (routing would happen here using workflow_type)
    execution_id = f"projects/demo/locations/us-central1/workflows/OrderProcessing/executions/{execution_name}"

    return transform_response(execution_name, execution_id)
```

---

## How the Routing Logic Works

By structuring the code this way, you've created a reliable dispatch system:

* **Backward Compatibility:** If an old client sends a request without a `workflow_type`, the API defaults to `"order"`, ensuring nothing breaks.
* **Security & Validation:** By using a "whitelist" (`allowed_types`), you prevent users from injecting unexpected strings into your execution names or internal logic.
* **Contextual Data:** The `workflow_input` dictionary allows the target workflow to know exactly what its "mission" is as soon as it receives the data.

### Example Request/Response

If you send this JSON:
```json
{
  "customerId": "CUST-99",
  "workflow_type": "refund",
  "amount": 50.00
}
```

The API will generate a name like `refund-a1b2c3d4` and return a **202 Accepted** status, indicating the refund process has officially started in the background.